# Simulating the UCJ ansatz with fermionic backpropagation

This guide demonstrates how to simulate a one-layer [UCJ ansatz](../explanations/lucj.ipynb) with fermionic backpropagation. This method can be used to efficiently simulate any one-layer UCJ ansatz exactly, with an optional final orbital rotation. It computes the expectation value of a one/two-body operator in the Heisenberg picture in $O(N^7)$ time for $N$ spatial orbitals.

In [17]:
import warnings
from collections import defaultdict

import pyscf
import pyscf.cc

import numpy as np

import scipy

import ffsim
from ffsim.variational.ucj_energy import ucj_energy, optimize_ucj_energy

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"


# LUCJ ansatz for a closed-shell molecule
We'll construct the ansatz for a nitrogen molecule in the 6-31g basis set. Since it's a closed-shell system, use the spin-balanced UCJ ansatz. We will restrict the pair connectivity, however fermionic backpropagation in general will work for any connectivity.

In [18]:
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()

n_reps = 1

# Define interactions
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = (
    None
)

# Use the backend implementable pairs_ab to construct the ucj_op
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization.
    # Additionally, you may want to set the multi_stage_start or multi_stage_step
    # arguments (or both) to obtain a better result at increased computational cost.
    # See the API documentation for details.
    optimize=True,
    options=dict(maxiter=100),
)


WARN: Unable to to identify input symmetry using original axes.
Different symmetry axes will be used.

converged SCF energy = -108.835236570774
norb = 16
nelec = (5, 5)
E(CCSD) = -109.0398256929733  E_corr = -0.2045891221988313


Now, let's simulate the ansatz using the backpropagation method. 

In [19]:
energy = ucj_energy(ucj_op, mol_hamiltonian, nelec, (pairs_aa, pairs_ab))
print(f"HF energy: {scf.e_tot:.6f}")
print(f"LUCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

HF energy: -108.835237
LUCJ Energy: -107.945197
CCSD energy: -109.039826


We can also variationally optimize the ansatz parameters to achieve lower ground state energies. This is particularly useful for strongly correlated systems where the CCSD parameters may not be optimal. This optimization is carried out using `optimize_ucj_energy`, which uses JAX for autodifferentiation. Note that for these examples, the number of terms is manageable on a GPU, but for larger systems, GPU acceleration might necessitate chunking the tensors, which can be set using the `chunk_size` argument. 

In [20]:
info = defaultdict(list)

def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")

optimal_ucj, result = optimize_ucj_energy(ucj_op, mol_hamiltonian, nelec, interaction_pairs=(pairs_aa, pairs_ab), return_optimize_result=True, callback=callback, options=dict(maxiter=10))

 -108.529320
 -108.703734
 -108.801863
 -108.812933
 -108.827696
 -108.830057
 -108.833662
 -108.837100
 -108.843061
 -108.850717


# UCJ ansatz for an open-shell molecule
We'll use a hydroxyl radical in the 6-31g basis set as an example of an open-shell system. For this example, we'll use unrestricted pair connectivity, i.e. the full UCJ ansatz.

In [21]:
# Build HO molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["H", (0, 0, 0)], ["O", (0, 0, 1.1)]],
    basis="6-31g",
    spin=1,
    symmetry="Coov",
)

# Get molecular data and Hamiltonian
scf = pyscf.scf.ROHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(scf).run()

# Use the backend implementable pairs_ab to construct the ucj_op
ucj_op = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=1,
    # Setting optimize=True enables the "compressed" factorization.
    # Additionally, you may want to set the multi_stage_start or multi_stage_step
    # arguments (or both) to obtain a better result at increased computational cost.
    # See the API documentation for details.
    optimize=True,
    options=dict(maxiter=100),
)

SCF not converged.
SCF energy = -75.3484557081271
norb = 11
nelec = (5, 4)

WARN: RCCSD method does not support ROHF method. ROHF object is converted to UHF object and UCCSD method is called.

E(UCCSD) = -75.45619739106766  E_corr = -0.1077416829405565


In [22]:
energy = ucj_energy(ucj_op, mol_hamiltonian, nelec)
print(f"HF energy: {scf.e_tot:.6f}")
print(f"UCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

HF energy: -75.348456
UCJ Energy: -75.411521
CCSD energy: -75.456197


In [23]:
info = defaultdict(list)

def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")

optimal_ucj, result = optimize_ucj_energy(ucj_op, mol_hamiltonian, nelec,  return_optimize_result=True, callback=callback, options=dict(maxiter=10))

 -75.422683
 -75.426638
 -75.427874
 -75.431436
 -75.434833
 -75.436450
 -75.438500
 -75.439623
 -75.441011
 -75.442355
